# 04 — Exploratory Data Analysis (EDA)

Notebook tạo và khám phá bộ dữ liệu Smart Parking 15.000 bản ghi. Dữ liệu được sinh theo các quy luật có liên hệ giữa ngữ cảnh vào bãi và hành vi đỗ xe để các mô hình có thể học được.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

np.random.seed(42)
rng = np.random.default_rng(42)

os.makedirs('../data/raw', exist_ok=True)
file_path = '../data/raw/parking_data_raw.csv'

print("Sẵn sàng tạo dataset Smart Parking.")

In [ ]:
# Tạo dữ liệu có quan hệ hợp lý giữa feature và target.
# QUAN TRỌNG: parking_behavior được quyết định từ ngữ cảnh vào bãi,
# sau đó duration_minutes được sinh theo behavior.
# Vì vậy Classification có thể học mà không cần dùng duration_minutes.

N = 15000

student_ids = np.array([f"SV{str(i).zfill(4)}" for i in range(1, 2001)])
students = rng.choice(student_ids, size=N)

start_date = pd.Timestamp('2023-09-01')
days = rng.integers(0, 121, size=N)
base_dates = start_date + pd.to_timedelta(days, unit='D')

is_morning = rng.random(N) < 0.65
entry_hour_float = np.where(
    is_morning,
    rng.normal(loc=7.0, scale=0.7, size=N),
    rng.normal(loc=13.0, scale=1.8, size=N)
)
entry_hour_float = np.clip(entry_hour_float, 5.5, 18.0)

entry_hour = np.floor(entry_hour_float).astype(int)
entry_minute = np.floor((entry_hour_float - entry_hour) * 60).astype(int)

entry_time = (
    base_dates
    + pd.to_timedelta(entry_hour, unit='h')
    + pd.to_timedelta(entry_minute, unit='m')
)

weekdays = entry_time.dayofweek
is_weekend = (weekdays >= 5)
is_exam_week = (entry_time.month == 12) & (entry_time.day > 10)

vehicle_type = rng.choice(
    ['Motorbike', 'Car'],
    size=N,
    p=[0.98, 0.02]
)

usual_zone = rng.choice(
    ['Zone_A', 'Zone_B', 'Zone_C', 'Zone_D'],
    size=N,
    p=[0.40, 0.30, 0.20, 0.10]
)

# Điểm cho 4 hành vi.
# Các feature đầu vào ảnh hưởng đến hành vi, nhưng duration chưa được sử dụng.
scores = np.column_stack([
    # Early_Return
    0.35
    + 0.80 * is_weekend
    + 0.45 * (entry_hour_float >= 12)
    + 0.20 * (usual_zone == 'Zone_A')
    - 0.30 * is_exam_week,

    # Full_Day
    1.00
    + 0.55 * (~is_weekend)
    + 0.45 * (entry_hour_float < 10)
    - 0.35 * (entry_hour_float >= 12)
    - 0.20 * (usual_zone == 'Zone_D'),

    # Overnight
    -1.35
    + 0.90 * is_exam_week
    + 0.75 * (usual_zone == 'Zone_D')
    + 0.20 * (entry_hour_float >= 12)
    + 0.15 * is_weekend,

    # Long_Term
    -2.15
    + 1.00 * is_exam_week
    + 0.95 * (usual_zone == 'Zone_D')
    + 0.20 * (entry_hour_float >= 12)
    + 0.15 * is_weekend
    + 0.10 * (vehicle_type == 'Car')
])

scores = scores + rng.normal(0, 0.15, size=scores.shape)

exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)

behaviors = np.array(['Early_Return', 'Full_Day', 'Overnight', 'Long_Term'])

behavior_indices = np.array([
    rng.choice(4, p=probabilities[i])
    for i in range(N)
])

parking_behavior = behaviors[behavior_indices]

# Duration được sinh sau khi đã xác định behavior.
duration_ranges = {
    'Early_Return': (60, 210),
    'Full_Day': (240, 540),
    'Overnight': (720, 1440),
    'Long_Term': (1440, 4320)
}

duration_minutes = np.array([
    int(rng.uniform(*duration_ranges[behavior]))
    for behavior in parking_behavior
])

exit_time = entry_time + pd.to_timedelta(duration_minutes, unit='m')

df = pd.DataFrame({
    'student_id': students,
    'vehicle_type': vehicle_type,
    'entry_time': entry_time,
    'exit_time': exit_time,
    'day_of_week': entry_time.day_name(),
    'duration_minutes': duration_minutes,
    'usual_zone': usual_zone,
    'is_exam_week': is_exam_week.astype(int),
    'parking_behavior': parking_behavior
})

df.to_csv(file_path, index=False)

print(f"Đã tạo {len(df):,} dòng dữ liệu.")
print(f"Đã lưu: {file_path}")

In [ ]:
print("Kích thước dataset:", df.shape)

print("
Kiểu dữ liệu:")
print(df.dtypes)

print("
Phân bố parking_behavior:")
print(df['parking_behavior'].value_counts())

print("
Tỷ lệ parking_behavior (%):")
print((df['parking_behavior'].value_counts(normalize=True) * 100).round(2))

print("
5 dòng đầu:")
display(df.head())

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='parking_behavior', order=df['parking_behavior'].value_counts().index)
plt.title('Phân bố hành vi đỗ xe')
plt.xlabel('Parking behavior')
plt.ylabel('Số lượng')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
summary = (
    df.groupby('parking_behavior')['duration_minutes']
      .agg(['count', 'mean', 'median', 'min', 'max'])
      .sort_values('count', ascending=False)
)

display(summary.round(2))

print("
Kiểm tra quan hệ giữa hành vi và các feature:")
display(pd.crosstab(df['parking_behavior'], df['is_exam_week'], normalize='index').round(3))
display(pd.crosstab(df['parking_behavior'], df['usual_zone'], normalize='index').round(3))